In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.feature import hog
from skimage import exposure
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
dataset_root = '/kaggle/input/vietnamese-foods'
image_root = os.path.join(dataset_root, 'Images')


train_dir = os.path.join(image_root, 'Train')
val_dir = os.path.join(image_root, 'Validate')
test_dir = os.path.join(image_root, 'Test')

image_size = (128, 128) # Resize ảnh để HOG hoạt động đồng bộ

def load_data(data_dir):
    X = []
    y = []
    # Kiểm tra đường dẫn có tồn tại không
    if not os.path.exists(data_dir):
        print(f"Lỗi: Không tìm thấy đường dẫn {data_dir}")
        return np.array([]), np.array([])

    classes = sorted(os.listdir(data_dir)) # Lấy danh sách folder (nhãn)
    print(f"Đang tải dữ liệu từ: {data_dir}")
    
    for label in tqdm(classes):
        path = os.path.join(data_dir, label)
        if not os.path.isdir(path):
            continue
            
        for img_name in os.listdir(path):
            img_path = os.path.join(path, img_name)
            try:
                # Đọc ảnh
                img = cv2.imread(img_path)
                if img is not None:
                    # Resize ảnh
                    img = cv2.resize(img, image_size)
                    X.append(img)
                    y.append(label)
            except Exception as e:
                pass
                
    return np.array(X), np.array(y)

# 1. Load Train và Validate
print("--- Bắt đầu Load dữ liệu ---")
X_train_part, y_train_part = load_data(train_dir)
X_val_part, y_val_part = load_data(val_dir)

# 2. Gộp Train và Val để GridSearch có nhiều dữ liệu hơn (CV sẽ tự chia lại sau)
if len(X_train_part) > 0 and len(X_val_part) > 0:
    X_train = np.concatenate((X_train_part, X_val_part), axis=0)
    y_train = np.concatenate((y_train_part, y_val_part), axis=0)
else:
    print("Cảnh báo: Không load được dữ liệu Train hoặc Validate. Hãy kiểm tra lại đường dẫn.")
    X_train, y_train = X_train_part, y_train_part

# 3. Load Test
X_test, y_test = load_data(test_dir)

print(f"\nTổng số ảnh Train (đã gộp Val): {len(X_train)}")
print(f"Tổng số ảnh Test: {len(X_test)}")

In [ ]:
class BGR2RGBTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        # Chuyển từ BGR (mặc định của cv2) sang RGB
        return np.array([cv2.cvtColor(img, cv2.COLOR_BGR2RGB) for img in X])

class HOGTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, orientations=9, pixels_per_cell=(16, 16), 
                 cells_per_block=(2, 2), transform_sqrt=True):
        self.orientations = orientations
        self.pixels_per_cell = pixels_per_cell
        self.cells_per_block = cells_per_block
        self.transform_sqrt = transform_sqrt

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        def local_hog(img):
            return hog(img,
                       orientations=self.orientations,
                       pixels_per_cell=self.pixels_per_cell,
                       cells_per_block=self.cells_per_block,
                       transform_sqrt=self.transform_sqrt,
                       block_norm='L2-Hys',
                       channel_axis=-1,
                       feature_vector=True)
        return np.array([local_hog(img) for img in X])

print("Đã khởi tạo Class HOG Transformer")

In [ ]:
# 1. Mã hóa nhãn (Label Encoding)
le = LabelEncoder()
# Fit trên cả tập train và test để đảm bảo bao phủ hết các nhãn (phòng trường hợp data thiếu)
all_labels = np.concatenate((y_train, y_test))
le.fit(all_labels)

y_train_encoded = le.transform(y_train)
y_test_encoded = le.transform(y_test)

print(f"Đã mã hóa {len(le.classes_)} nhãn món ăn.")

# 2. Pipeline
model_pipeline = Pipeline([
    ('rgb', BGR2RGBTransformer()),
    ('hog', HOGTransformer()),
    ('svm', SVC(C=10, gamma='scale', kernel='rbf')) # Tham số bạn yêu cầu
])

print("Đang huấn luyện mô hình với tham số cố định...")
model_pipeline.fit(X_train, y_train_encoded)
print("Huấn luyện hoàn tất.")

In [ ]:
# --- 3. DỰ ĐOÁN VÀ ĐÁNH GIÁ CƠ BẢN ---
print("Đang dự đoán trên tập Test...")
y_pred = model_pipeline.predict(X_test)

# Chuyển ngược lại từ số sang tên món ăn
y_test_names = le.inverse_transform(y_test_encoded)
y_pred_names = le.inverse_transform(y_pred)

# Tính toán các chỉ số
acc = accuracy_score(y_test_names, y_pred_names)
prec = precision_score(y_test_names, y_pred_names, average='weighted')
rec = recall_score(y_test_names, y_pred_names, average='weighted')
f1 = f1_score(y_test_names, y_pred_names, average='weighted')

print("\n--- KẾT QUẢ ĐÁNH GIÁ TỔNG HỢP ---")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}")

print("\n--- CHI TIẾT TỪNG LỚP (CLASSIFICATION REPORT) ---")
print(classification_report(y_test_names, y_pred_names))

In [ ]:
# --- 4. VẼ MA TRẬN NHẦM LẪN (CONFUSION MATRIX) DẠNG PHẦN TRĂM ---
import matplotlib.pyplot as plt
import seaborn as sns

# Tính ma trận nhầm lẫn
cm = confusion_matrix(y_test_names, y_pred_names, labels=le.classes_)

# Chuyển đổi sang phần trăm (Normalize theo hàng - True Labels)
# Cộng thêm 1e-9 để tránh lỗi chia cho 0 nếu có lớp không có mẫu nào trong test
cm_percent = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-9)

# Code vẽ biểu đồ theo yêu cầu của bạn
plt.figure(figsize=(30, 30))
plt.imshow(cm_percent, interpolation='nearest', cmap=plt.cm.Blues)
plt.title("Confusion Matrix (%)")
plt.colorbar()

tick_marks = np.arange(len(le.classes_))
plt.xticks(tick_marks, le.classes_, rotation=90, fontsize=12)
plt.yticks(tick_marks, le.classes_, fontsize=12)

# Hiển thị số phần trăm trong từng ô
thresh = cm_percent.max() / 2
for i in range(cm_percent.shape[0]):
    for j in range(cm_percent.shape[1]):
        # Chỉ hiển thị text nếu giá trị > 0.001 (0.1%) để đỡ rối mắt với các ô 0%
        if cm_percent[i, j] > 0.001: 
            plt.text(j, i, f"{cm_percent[i, j]*100:.1f}%",
                     horizontalalignment="center",
                     verticalalignment="center",
                     color="white" if cm_percent[i, j] > thresh else "black",
                     fontsize=10)

plt.xlabel("Predicted Label", fontsize=14)
plt.ylabel("True Label", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import joblib
joblib.dump(model_pipeline, 'best_vietfood_model_fixed.pkl')
print("Đã lưu model thành công: best_vietfood_model_fixed.pkl")